## Coding Exercise #0602B

### 1. Classification:

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import *

print(tf.__version__)

#### 1.1. Data

In [ ]:
# Read data.
dataset = pd.read_csv('Mobile_Customers.csv')
dataset.head()

#### 1.2. Data Cleansing

In [ ]:
# Check number of NaN values.
dataset.isna().sum()

In [ ]:
# Remove unnecessary columns: id
dataset.drop(columns=['id'], inplace=True)

In [ ]:
# Remove columns with too many NaN values: college
dataset.drop(columns=['college'], inplace=True)

# Remove rows with NaN values
dataset.dropna(inplace=True)

In [ ]:
# Check number of NaN values.
dataset.isna().sum()

In [ ]:
# Dummy variableization.
dataset = pd.get_dummies(dataset, columns=['satisfaction', 'usage_level', 'change_plan'], drop_first=True)
dataset.head()

#### 1.3. Divide data into the training set and the test set

In [ ]:
# Separate data for training and testing.
train_dataset = dataset.sample(frac=0.8, random_state=0)
test_dataset = dataset.drop(train_dataset.index)

#### 1.4. Check the statistics

In [ ]:
# Check statistical information
train_dataset.describe()

#### 1.5. Split target values, or "labels", in feature

In [ ]:
# Target separation.
train_labels = train_dataset.pop('brand_churn')
test_labels = test_dataset.pop('brand_churn')

#### 1.6. Normalize data

In [ ]:
# Normalization.
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

normed_train_data = scaler.fit_transform(train_dataset)
normed_test_data = scaler.transform(test_dataset)

#### 1.7. Create the model

In [ ]:
# Make the sequential model with densely connected layer. The output layer returns the consecutive values.  

In [ ]:
# Create a function to create a model.
def build_model():
    model = Sequential()
    model.add(Input(shape=(len(train_dataset.keys()), )))
    model.add(Dense(8, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])
    return model

# Build a model.
model = build_model()

#### 1.8. Check the model

In [ ]:
# Check model.
model.summary()

#### 1.9. Train the model

In [ ]:
# Model training.
history = model.fit(normed_train_data, train_labels, epochs=100, validation_split=0.2, verbose=0)

In [ ]:
# Check learning information.
hist = pd.DataFrame(history.history)
hist['epoch'] = history.epoch
hist.head()

In [ ]:
# Creating a learning process visualization function
def plot_history(history):
    hist = pd.DataFrame(history.history)
    hist['epoch'] = history.epoch
    
    plt.figure(figsize=(8,12))
    
    plt.subplot(2, 1, 1)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.plot(hist['epoch'], hist['loss'], label='Train Loss')
    plt.plot(hist['epoch'], hist['val_loss'], label='Val Loss')
    plt.legend()
    plt.show()

# Visualization.
plot_history(history)

In [ ]:
# By modifying the model.fit method, let trainig automatically stop when varification does not improve.  
# Use EarlyStopping callback to check the training process at every epoch.
# Automatically stop training when there is no more improvement over the specified number of epochs.  

In [ ]:
# Rebuild the model.
model = build_model()

# The 'patience' parameter is the epoch number to be used for checkking improvement. 
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)

history = model.fit(normed_train_data, train_labels, epochs=100,
                    validation_split=0.2, verbose=0, callbacks=[early_stop])

plot_history(history)

#### 1.10. Prediction

In [ ]:
# Predict.
test_predictions = model.predict(normed_test_data)
print(test_predictions[:5])

In [ ]:
# Probability value => Convert to 0, 1
test_predictions = np.where(test_predictions >= 0.5, 1, 0)
print(test_predictions[:5])

In [ ]:
# Performance evaluation.
print(classification_report(test_labels, test_predictions))